In [1]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "/mnt/SSD-playing games/Workspace/obsidian_aio/Notebook/Dự án/OCR anh hiếu/fine-tunning/bestcheckpoint/turn2/checkpoint-40",
    max_seq_length = 5096, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = True, # [NEW!] A bit more accurate, uses 2x memory
    # token = "hf_...", # use one if using gated models
)

model.eval()  # Set the model to evaluation mode

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 07-31 21:18:37 [__init__.py:235] Automatically detected platform cuda.
==((====))==  Unsloth 2025.7.8: Fast Llama patching. Transformers: 4.53.3. vLLM: 0.10.1.dev73+g7728dd77b.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.472 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32+8ed0992.d20250726. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
import os 
from transformers import TextStreamer
from prompt_cached import get_prompt
def read_txt(file_path: str) -> str:
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()
    return content

for filename in os.listdir('./testset'):
    if filename.endswith('pdf'): continue
    
    # messages = [{
    #     "role": "system",
    #     "content": [{
    #         "type" : "text",
    #         "text" : get_prompt()
    #     }],
    #     "role": "user",
    #     "content": [{
    #         "type" : "text",
    #         "text" : f"\n**TÀI LIỆU MARKDOWN**:\n{read_txt(os.path.join('./testset', filename))}"
    #     }],

    # }]
    print("-"*50, filename, "-"*50)
    messages = [
        {
            "role": "system", 
            "content": get_prompt()
        },
        {
            "role": "user", 
            "content": f"\n**TÀI LIỆU MARKDOWN**:\n{read_txt(os.path.join('./testset', filename))}"
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt = True, # Must add for generation
    )
    _ = model.generate(
        **tokenizer([text], return_tensors = "pt").to("cuda"),
        max_new_tokens = 5096, # Increase for longer outputs!
        # Recommended Gemma-3 settings!
        temperature = 0.3, top_p = 0.95, top_k = 64, num_beams = 1, do_sample = True,
        
        streamer = TextStreamer(tokenizer, skip_prompt = True),
    )
    print("-"*50, "END", "-"*50)

-------------------------------------------------- 1587366867.txt --------------------------------------------------
<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

Bạn là một trợ lý giúp tôi trích xuất thông tin từ tài liệu Markdown và tạo JSON chính xác theo schema. Tất cả các trường dưới đây PHẢI xuất hiện trong JSON đầu ra.

**Nhiệm vụ**: 
**Các trường BẮT BUỘC phải có**:
1. `MaHoSo`: Mã hồ sơ (string)
2. `SoDon`: Số đơn đăng ký (string)
3. `SoDangKyLanDau`: Số đăng ký lần đầu (string)
4. `LoaiDonID`: ID loại đơn (integer, ánh xạ enum)
5. `LoaiDonName`: Tên loại đơn (string)
6. `LoaiDonCode`: Mã loại đơn (string)
7. `LoaiHinhGDID`: ID loại hình giao dịch (integer, ánh xạ enum)
8. `LoaiHinhGDName`: Tên loại hình giao dịch (string)
9. `LoaiBienPhapID`: ID biện pháp bảo đảm (integer, ánh xạ enum)
10. `LoaiBienPhapName`: Tên biện pháp bảo đảm (string)
11. `LoaiHopDongID`: ID loại hợp đồng (int